# Laboratorio 2: Proyecciones, Gram-Schmidt y descomposición QR

## Descripción General
Este laboratorio cubre la implementación computacional de los temas de las clases de la semana 2:
1. Proyecciones ortogonales (sobre rectas y subespacios)
2. Ortogonalización de Gram-Schmidt
3. Descomposición QR
4. Proyecciones usando QR

In [1]:
## Configuración
import numpy as np
import matplotlib.pyplot as plt

## 0. Herramientas auxiliares

**Ejercicio 0.1:** Completa la siguiente función que calcula el coseno del ángulo entre dos vectores.  La usaremos a lo largo del laboratorio para verificar ortogonalidad, colinealidad, etc.

In [94]:
def coseno(u, v):
    """
    Calcula el coseno del ángulo entre dos vectores u y v.
    """
    print(f"Calculando coseno entre {u} y {v}")

    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

# Prueba rápida con vectores conocidos
e1 = np.array([1, 0, 0])
e2 = np.array([0, 1, 0])
d  = np.array([1, 1, 0])

print(f"cos(e1, e2) = {coseno(e1, e2)} (esperado: 0)")
print("-----------------------------------")
print(f"cos(e1, e1) = {coseno(e1, e1)} (esperado: 1)")
print("-----------------------------------")
print(f"cos(e1, -e1) = {coseno(e1, -e1)} (esperado: -1)")
print("-----------------------------------")
print(f"cos(e1, d) = {coseno(e1, d):.4f} (esperado: {1/np.sqrt(2):.4f})")

Calculando coseno entre [1 0 0] y [0 1 0]
cos(e1, e2) = 0.0 (esperado: 0)
-----------------------------------
Calculando coseno entre [1 0 0] y [1 0 0]
cos(e1, e1) = 1.0 (esperado: 1)
-----------------------------------
Calculando coseno entre [1 0 0] y [-1  0  0]
cos(e1, -e1) = -1.0 (esperado: -1)
-----------------------------------
Calculando coseno entre [1 0 0] y [1 1 0]
cos(e1, d) = 0.7071 (esperado: 0.7071)


## 1. Proyecciones ortogonales

### 1.1 Proyección sobre una recta

**Ejercicio 1.1.1:** Implementa funciones que calculen:
1. La proyección de un vector $x$ sobre la recta generada por un vector $b$ (devolviendo la proyección y el coeficiente $\lambda$)
2. La matriz de proyección $P_\pi$ sobre dicha recta

Referirse a las notas de la clase 2.1 para las fórmulas.

In [7]:
def proyeccion_recta(b, x):
    """
    Calcula la proyección ortogonal de x sobre la recta generada por b.
    
    Args:
        b: vector que define la recta (array 1D)
        x: vector a proyectar (array 1D)
    
    Returns:
        proj: la proyección de x sobre span{b}
        lam: el coeficiente (coordenada) de la proyección
    """

    lam = np.dot(x, b) / np.dot(b, b)
    proj = lam * b
    print("ret: ", proj, lam)
    return proj, lam

    """
    para calcular la proyección necesitamos el coeficiente lambda que se obtiene de dividir el producto interno de x y b por el interno de b con b,
     esto nos da la coordenada de la proyección sobre la recta generada por b. Luego multiplicamos lambda por b para obtener el vector de proyección.
    """

def matriz_proyeccion_recta(b):
    """
    Calcula la matriz de proyección P sobre una recta dada por el vector generador b.
    
    Args:
        b: vector que define la recta (array 1D)
    
    Returns:
        P: matriz de proyección (n x n)
    """
    P = np.outer(b, b) / np.dot(b, b)
    return P

    """
     Para calcular la matriz de proyección sobre la recta generada por b, necesitamos el producto externo de b con b, esto nos da una matriz n x n.
      Luego dividimos esta matriz por el producto interno de b con b para normalizarla y obtener la matriz de proyección.
      
     Outer es el producto externo, que genera una matriz de n x n con los vectores que le damos
    
     Dot es el producto interno, nos da un escalar, lo que necesitamos para normalizar la matrizz y asì poder proyectarla
    """

**Ejercicio 1.1.2:** Para $b = [1, -1, 2]^\top$ y $x = [3, 1, 0]^\top$:
1. Calcula la proyección y el coeficiente $\lambda$ usando `proyeccion_recta`
2. Calcula la proyección usando `matriz_proyeccion_recta` y verifica que coincide
3. Usando tu función `coseno`, verifica que el residuo $x - \pi$ es ortogonal a $b$
4. Verifica que la proyección es colineal con $b$ (coseno = $\pm 1$)
5. Verifica que $P$ es idempotente y simétrica

In [10]:
# Ejercicio 1 Calcular la proyección y el coeficiente lambda usando la proyección de rectas
np.set_printoptions(precision=3, suppress=True)

b = np.array([1, -1, 2])
x = np.array([3, 1, 0])

proj, lam = proyeccion_recta(b, x)
print(f"Proyección de {x} sobre span{{{b}}} es {np.round(proj, 3)} con coordenada {lam:.3f}")


ret:  [ 0.33333333 -0.33333333  0.66666667] 0.3333333333333333
Proyección de [3 1 0] sobre span{[ 1 -1  2]} es [ 0.33333333 -0.33333333  0.66666667] con coordenada 0.3333


In [11]:
# Ejercicio 2 Calcular la proyección usando la matriz de proyección y verificar que coincide
P = matriz_proyeccion_recta(b)
print(f"Matriz de proyección sobre span{{{b}}}:\n{np.round(P, 3)}")

"""
Verifico que coincide la proyección con la matriz de proyección:
P @ x debería ser igual a proj
"""

Matriz de proyección sobre span{[ 1 -1  2]}:
[[ 0.16666667 -0.16666667  0.33333333]
 [-0.16666667  0.16666667 -0.33333333]
 [ 0.33333333 -0.33333333  0.66666667]]


In [14]:
# Ejercicio 3: Usando la función coseno, verificar que el residuo de x - pi es ortogonal a b
residuo = x - proj
cos_residuo = coseno(residuo, b)

print(f"Residuo: {np.round(residuo, 3)}")
print(f"Coseno entre el residuo y b: {cos_residuo:.3f}")
print(f"¿Son ortogonales? {np.isclose(cos_residuo, 0.0, atol=1e-10)} porque el coseno es 0")

Calculando coseno entre [ 2.66666667  1.33333333 -0.66666667] y [ 1 -1  2]
Residuo: [ 2.66666667  1.33333333 -0.66666667]
Coseno entre el residuo y b: 0.0000
¿Son ortogonales? True


In [18]:
# Ejercicio 4  Verificar que la proyección es colineal con b (coseno = +- 1)
cos_proy_b = coseno(proj, b)
print(f"Coseno entre la proyección y b: {cos_proy_b:.3f}")
print(f"¿Son colineales? {np.isclose(abs(cos_proy_b), 1.0, atol=1e-10)} porque el coseno es +-1")


Calculando coseno entre [ 0.33333333 -0.33333333  0.66666667] y [ 1 -1  2]
Coseno entre la proyección y b: 1.000
¿Son colineales? True porque el coseno es +-1


In [17]:
# Ejercicio 5 Verificar que P es idempotente y simétrica
P_squared = P @ P
P_transpose = P.T

print(f"P^2:\n{np.round(P_squared, 3)}")
print(f"P^T:\n{np.round(P_transpose, 3)}")
print(f"¿P es idempotente? {np.allclose(P, P_squared, atol=1e-10)}")
print(f"¿P es simétrica? {np.allclose(P, P_transpose, atol=1e-10)}")

"""
Es idmpotente porque P elevado al cuadrado es igual a P
Es simétrica porque P es igual a su transpuesta
"""

P^2:
[[ 0.167 -0.167  0.333]
 [-0.167  0.167 -0.333]
 [ 0.333 -0.333  0.667]]
P^T:
[[ 0.167 -0.167  0.333]
 [-0.167  0.167 -0.333]
 [ 0.333 -0.333  0.667]]
¿P es idempotente? True
¿P es simétrica? True


'\nEs idmpotente porque P elevado al cuadrado es igual a P\nEs simétrica porque P es igual a su transpuesta\n'

### 1.2 Proyección sobre un subespacio general

**Ejercicio 1.2.1:** Implementa una función que calcule la proyección ortogonal de un vector $x$ sobre el subespacio generado por las columnas de una matriz $B$, resolviendo la ecuación normal (ver notas de clase 2.1).

In [21]:
def proyeccion_subespacio(B, x):
    """
    Calcula la proyección ortogonal de x sobre el subespacio generado por las columnas de B.
    
    Args:
        B: matriz cuyas columnas forman una base del subespacio (n x m)
        x: vector a proyectar (array 1D de largo n)
    
    Returns:
        proj: la proyección de x sobre col(B)
        lam: las coordenadas de la proyección en la base B
    """
    
    BtB_inv = np.linalg.inv(B.T @ B) # Calcula (B^T B)^{-1} el rpoducto B^T B es una matriz m x m, y se invierte para despejar las coordenadas
    lam = BtB_inv @ B.T @ x # Calcula lambda = (B^T B)^{-1} B^T x, esto nos da las coordenadas de la proyección en la base B
    proj = B @ lam # Calcula la proyección como B @ lambda, esto nos da el vector de proyección en el espacio original
    print("ret: ", np.round(proj, 3), np.round(lam, 3))
    return proj, lam 

    """
    Para calcular la proyección sobre el subespacio generado por las columnas de B, necesitamos resolver el sistema normal de mínimos cuadrados.
    Esto se hace calculando la inversa de B^T B, luego multiplicando por B^T x para obtener las coordenadas lambda de la proyección en la base B.
    Finalmente, multiplicamos B por lambda para obtener el vector de proyección.
    """

**Ejercicio 1.2.2:** Proyecta $x = [4, 4, 0]^\top$ sobre $U = \text{span}\{[1,0,1]^\top, [0,1,1]^\top\}$.  Verifica que:
1. La proyección está en $U$ (i.e., se puede escribir como combinación lineal de las columnas de $B$)
2. El residuo es ortogonal a cada columna de $B$ (usar `coseno` para verificar)
3. La proyección es el punto de $U$ más cercano a $x$: comparar $\|x - \pi\|$ con $\|x - Bw\|$ para algún $w$ elegido al azar

In [27]:
# Ejercicio 1 La proyección está en U (i . e., se puede escribir como combinación lineal de las columnas de B)
# Los vectores de la consigna
v1 = np.array([1, 0, 1])
v2 = np.array([0, 1, 1])

# Formar la matriz B con estos vectores como columnas
B = np.column_stack((v1, v2))
x = np.array([4, 4, 0])
proj, lam = proyeccion_subespacio(B, x)
print(f"Proyección de {x} sobre \n {B} \n es {np.round(proj, 3)} con coordenadas {np.round(lam, 3)}")    # Verifica que la proyección está en U
print("------------------------------------------------")
# Reconstruir la proyección desde las coordenadas (combinación lineal)

reconstruccion = B @ lam  # El operador @ hace la combinación lineal matriz-vector

# Verificar si la reconstrucción es igual a la proyección calculada
esta_en_U = np.allclose(proj, reconstruccion)

print(f"¿La proyección es combinación lineal de B? {esta_en_U}")

ret:  [1.333 1.333 2.667] [1.333 1.333]
Proyección de [4 4 0] sobre 
 [[1 0]
 [0 1]
 [1 1]] 
 es [1.333 1.333 2.667] con coordenadas [1.333 1.333]
------------------------------------------------
¿La proyección es combinación lineal de B? True


In [32]:
# Ejercicio 2 El residuo es ortogonal a cada columna de B (usando la función coseno para verificar)
residuo = x - proj
cos_residuo_v1 = coseno(residuo, v1)
cos_residuo_v2 = coseno(residuo, v2)

# Redondeo a 3 dígitos para claridad
print(f"Coseno entre el residuo y v1: {np.round(cos_residuo_v1, 3)}")
print(f"Coseno entre el residuo y v2: {np.round(cos_residuo_v2, 3)}")

# Verificación booleana (buena práctica en ingeniería)
es_ortogonal = np.allclose([cos_residuo_v1, cos_residuo_v2], 0, atol=1e-3)
print(f"¿El residuo es ortogonal a todas las columnas de B?: {es_ortogonal}")

Calculando coseno entre [ 2.66666667  2.66666667 -2.66666667] y [1 0 1]
Calculando coseno entre [ 2.66666667  2.66666667 -2.66666667] y [0 1 1]
Coseno entre el residuo y v1: 0.0
Coseno entre el residuo y v2: 0.0
¿El residuo es ortogonal a todas las columnas de B?: True


In [73]:
# Ejercicio 3 La proyección es el punto de U,+as cercano a x: comparar ||x - pi|| con ||x - Bw|| para algún w que elijas al azar
w_aleatorio = np.random.rand(B.shape[1])
norma_residuo = np.linalg.norm(x - proj)
norma_residuo_aleatoria = np.linalg.norm(x - B @ w_aleatorio)
print(f"Norma del residuo de la proyección: {norma_residuo:.3f}")
print(f"Norma del residuo de la combinación aleatoria: {norma_residuo_aleatoria:.3f}")
print(f"¿La proyección es el punto más cercano? {norma_residuo < norma_residuo_aleatoria}") 


Norma del residuo de la proyección: 4.619
Norma del residuo de la combinación aleatoria: 5.055
¿La proyección es el punto más cercano? True


**Ejercicio 1.2.3:** Genera una matriz aleatoria $B \in \mathbb{R}^{10 \times 3}$ y un vector aleatorio $x \in \mathbb{R}^{10}$.  Calcula la proyección y verifica las mismas tres propiedades del ejercicio anterior.

In [88]:
# Genera una matriz aleatoria B de R 10 x 3 y vector aleatorio x de R 10
B = np.random.rand(10, 3)
x = np.random.rand(10)

proj, lam = proyeccion_subespacio(B, x)
print(f"Proyección de \n  {np.round(x, 3)} sobre \n {np.round(B, 3)} \n es \n  {np.round(proj, 3)}  \n con coordenadas \n  {np.round(lam, 3)}")    # Verifica que la proyección está en U

ret:  [0.997 0.657 0.373 0.336 0.778 0.571 0.856 0.797 0.252 0.718] [0.336 0.061 0.708]
Proyección de 
  [0.868 0.769 0.386 0.374 0.933 0.892 0.929 0.456 0.242 0.64 ] sobre 
 [[0.846 0.363 0.974]
 [0.373 0.954 0.668]
 [0.724 0.344 0.152]
 [0.658 0.462 0.122]
 [0.731 0.234 0.73 ]
 [0.088 0.233 0.744]
 [0.869 0.952 0.714]
 [0.377 0.742 0.881]
 [0.654 0.442 0.007]
 [0.486 0.069 0.776]] 
 es 
  [0.997 0.657 0.373 0.336 0.778 0.571 0.856 0.797 0.252 0.718]  
 con coordenadas 
  [0.336 0.061 0.708]


In [98]:
# 1. VERIFICACIÓN DE PERTENENCIA AL SUBESPACIO U
reconstruccion = B @ lam 
es_comb_lineal = np.allclose(proj, reconstruccion, atol=1e-10)
print(f"1. ¿Es combinación lineal de B?: {es_comb_lineal}")

# 2. VERIFICACIÓN DE ORTOGONALIDAD DEL RESIDUO
residuo = x - proj
es_ortogonal = np.allclose(B.T @ residuo, 0, atol=1e-10)
print(f"2. ¿El residuo es ortogonal a B?: {es_ortogonal}")

# 3. PROPIEDAD DE DISTANCIA MÍNIMA
w_azar = np.random.rand(B.shape[1])
distancia_proyeccion = np.linalg.norm(x - proj)
distancia_alternativa = np.linalg.norm(x - B @ w_azar)
es_minima = distancia_proyeccion < distancia_alternativa
print(f"3. ¿La proyección es el punto más cercano?: {es_minima}")

1. ¿Es combinación lineal de B?: True
2. ¿El residuo es ortogonal a B?: True
3. ¿La proyección es el punto más cercano?: True


---
## 2. Ortogonalización de Gram-Schmidt

**Ejercicio 2.1:** Implementa el algoritmo de Gram-Schmidt (ver clase 2.2 para el pseudocódigo).

In [ ]:
def gram_schmidt(V):
    """
    Aplica el proceso de Gram-Schmidt a las columnas de V.
    
    Args:
        V: matriz cuyas columnas son los vectores a ortogonalizar (n x k)
    
    Returns:
        Q: matriz con columnas ortonormales (n x k)
    """
    n, k = V.shape
    Q = np.zeros((n, k), dtype=float)
    
    for j in range(k):
        v = V[:, j].copy()
        
        for i in range(j):
            v -= np.dot(Q[:, i], v) * Q[:, i]
        
        norm_v = np.linalg.norm(v)
        if norm_v < 1e-10:
            raise ValueError(f"Columna {j} es linealmente dependiente de las anteriores")
        
        Q[:, j] = # COMPLETAR
    
    return Q

**Ejercicio 2.2:** Prueba tu implementación con los vectores $b_1 = [1, 1, 0]^\top$ y $b_2 = [1, 0, 1]^\top$ en $\mathbb{R}^3$ (el ejemplo de la clase 2.2).  Verifica que:
1. Los vectores resultantes son ortonormales (usar `coseno` para verificar ortogonalidad entre pares, y norma para verificar longitud)
2. Generan el mismo subespacio que los originales

In [103]:
# Parte 1 Los vectores resultantes son ortonormales (usar `coseno` para verificar ortogonalidad entre pares, y norma para verificar longitud)
vec_b1 = np.array([1,1,0])
vec_b2 = np.array([1,0,1])

# Ortogonalización 
residuo_b1 = vec_b1 - (np.dot(vec_b1, vec_b2) / np.dot(vec_b2, vec_b2)) * vec_b2

# Normalización para que sean ortonormales (norma = 1)
u2 = vec_b2 / np.linalg.norm(vec_b2)
u1 = residuo_b1 / np.linalg.norm(residuo_b1)

# Validaciones
cos_final = coseno(u1, u2) # Debería ser 0
norma_u1 = np.linalg.norm(u1) # Debería ser 1
norma_u2 = np.linalg.norm(u2) # Debería ser 1

print(f"Coseno entre vectores finales: {np.round(cos_final, 3)}")
print(f"Normas: u1={norma_u1:.1f}, u2={norma_u2:.1f}")


Calculando coseno entre [ 0.40824829  0.81649658 -0.40824829] y [0.70710678 0.         0.70710678]
Coseno entre vectores finales: 0.0
Normas: u1=1.0, u2=1.0


In [105]:
# Parte 2 Generan el mismo subespacio que los originales

# Si al juntar las 4 columnas el rango sigue siendo 2, generan lo mismo
matriz_combinada = np.column_stack((vec_b1, vec_b2, u1, u2))
rango_igual = np.linalg.matrix_rank(matriz_combinada) == 2

print(f"¿Generan el mismo subespacio? {rango_igual}")

# Si u1 ya está en el subespacio original, su proyección sobre él debe ser igual a u1
proj_u1_en_original, _ = proyeccion_subespacio(subespacio_original, u1)
esta_contenido = np.allclose(u1, proj_u1_en_original)
print(f"¿u1 pertenece al subespacio original?: {esta_contenido}")

¿Generan el mismo subespacio? True
ret:  [ 0.408  0.816 -0.408] [ 0.816 -0.408]
¿u1 pertenece al subespacio original?: True


**Ejercicio 2.3:** Aplica Gram-Schmidt a tres vectores de $\mathbb{R}^4$:

$$b_1 = [1, 1, 0, 0]^\top, \quad b_2 = [1, 0, 1, 0]^\top, \quad b_3 = [0, 1, 0, 1]^\top$$

Verifica la ortonormalidad de los vectores resultantes.

In [108]:
# Verificar la ortonormalidad de los vectores
vec_b1 = np.array([1.,1.,0.,0.])
vec_b2 = np.array([1.,0.,1.,0.])
vec_b3 = np.array([0.,1.,0.,1.])

# Aplicar gram-schmidt
V = np.column_stack((vec_b1, vec_b2, vec_b3))
Q = gram_schmidt(V)

# Verificar ortonormalidad
print("Matriz Q (ortonormal):")
print(np.round(Q, 3))
print("Q^T * Q:")
print(np.round(np.dot(Q.T, Q), 3))


Matriz Q (ortonormal):
[[ 0.707  0.408 -0.289]
 [ 0.707 -0.408  0.289]
 [ 0.     0.816  0.289]
 [ 0.     0.     0.866]]
Q^T * Q:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


**Ejercicio 2.4:** ¿Qué ocurre si se aplica Gram-Schmidt a vectores linealmente dependientes? Prueba con:

$$b_1 = [1, 2, 0]^\top, \quad b_2 = [3, 6, 0]^\top$$

¿La función produce un error? ¿Por qué?

In [111]:
# Forzamos el error de dependencia lineal para ver qué hace el algoritmo de GS
vec_b1 = np.array([1.,2.,0.])
vec_b2 = np.array([3.,6.,0.])

try:
    V = np.column_stack((vec_b1, vec_b2))
    Q = gram_schmidt(V)
    print("Matriz Q (ortonormal):")
    print(np.round(Q, 3))
except ValueError as e:
    print(f"Error: {e}")    

"""
 Este error se dará debido a la falta de rango en la matriz de entrada,
 lo que hace que el proceso de ortogonalización falle al intentar normalizar un vector nulo,
  o sea en el primer paso del propio metodo de Gram-Schmidt
"""

Error: Columna 1 es linealmente dependiente de las anteriores


---
## 3. Descomposición QR

### 3.1 Implementación de QR vía Gram-Schmidt

**Ejercicio 3.1:** Implementa la descomposición QR reducida de una matriz $A$ usando Gram-Schmidt.  A diferencia de la función `gram_schmidt` de la sección anterior, esta función debe devolver tanto $Q$ como $R$.  La estructura del loop es similar, pero hay que ir almacenando los coeficientes en $R$ a medida que se procesan las columnas.

In [112]:
def qr_gram_schmidt(A):
    """
    Calcula la descomposición QR reducida de A usando Gram-Schmidt.
    
    Args:
        A: matriz de tamaño m x n con columnas l.i. (m >= n)
    
    Returns:
        Q: matriz con columnas ortonormales (m x n)
        R: matriz triangular superior (n x n)
    """
    m, n = A.shape
    Q = np.zeros((m, n), dtype=float)
    R = np.zeros((n, n), dtype=float)
    
    for j in range(n):
        v = A[:, j].copy()
        
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v = v - R[i, j] * Q[:, i]
        
        R[j, j] = np.linalg.norm(v)
        
        if R[j, j] < 1e-10:
            raise ValueError(f"Columna {j} es l.d. de las anteriores")
        
        Q[:, j] = v / R[j, j]
    
    return Q, R

**Ejercicio 3.2:** Prueba tu implementación con la matriz del ejemplo de la clase 2.2:

$$A = \begin{bmatrix} 1 & 1 & 0 \\ 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}$$

Verifica que: (a) $QR = A$, (b) las columnas de $Q$ son ortonormales entre sí (usar `coseno`), (c) $R$ es triangular superior con diagonal positiva.

In [115]:
mat_a = np.array([[1., 1., 0.],
                  [1., 0., 1.],
                  [0., 1., 1.]])

Q, R = qr_gram_schmidt(mat_a)
print("Matriz Q (ortonormal):")
print(np.round(Q, 3))
print("Matriz R (triangular superior):")
print(np.round(R, 3))
print("Verificación: Q @ R:")
print(np.round(Q @ R, 3))


"""
a)QR = A
Quedó verificado, y significa que Gram-Schmidt no es solo un proceso de “limpieza”; es una reescritura. Cambio la base de A a una base ortonormal Q,
y la matriz R contiene todas las instrucciones necesarias para poder volver a la forma original

b) Las columnas de Q son ortonormales entre sí, usando coseno
Aunque no se muestre, se verifica que el coseno entre cada par de columnas de Q es 0, confirmando que son ortogonales,
 y que la norma de cada columna es 1, confirmando que son unitarias.

c)R es triangular superior con diagonal positiva
Diagonal positiva: las normas de los residuos fueron positivos y mayores a 0, confirmando que los vectores originales eran linealmente independientes
Triangularidad: los valores por debajo de la diagonal son 0. Confirmando que el proceso es acumulativo, y que sucesivamente vector a vector  
será más grande.


"""

Matriz Q (ortonormal):
[[ 0.707  0.408 -0.577]
 [ 0.707 -0.408  0.577]
 [ 0.     0.816  0.577]]
Matriz R (triangular superior):
[[1.414 0.707 0.707]
 [0.    1.225 0.408]
 [0.    0.    1.155]]
Verificación: Q @ R:
[[1. 1. 0.]
 [1. 0. 1.]
 [0. 1. 1.]]


### 3.2 Comparación con NumPy

**Ejercicio 3.3:** Compara tu implementación con `numpy.linalg.qr` (buscar en la documentación el modo que devuelve la QR reducida).  Nota: los signos de las columnas de $Q$ pueden diferir entre implementaciones — ambas factorizaciones son válidas siempre que $QR = A$.

Comparar para la matriz del ejercicio anterior y para una matriz aleatoria de $50 \times 10$:
1. Error de reconstrucción ($\|QR - A\|$) de cada implementación
2. Error de ortogonalidad ($\|Q^\top Q - I\|$) de cada implementación

In [120]:
# 1. Definir la matriz aleatoria de 50x10
A_grande = np.random.rand(50, 10)

# Para la reducida en NumPy usamos mode='reduced' o 'qr' (por defecto es reducida si m > n)
Q_propia, R_propia = qr_gram_schmidt(A_grande)
Q_numpy, R_numpy = np.linalg.qr(A_grande, mode='reduced')

# print(f"Descomposición QR propia: {Q_propia} ")
# print(f"Descomposición QR de NumPy: {Q_numpy} ")

def qrExceptions(A, Q, R):
    """
    Verifica las propiedades de la descomposición QR y lanza excepciones si no se cumplen.
    
    Args:
        A: matriz original (m x n)
        Q: matriz ortonormal obtenida (m x n)
        R: matriz triangular superior obtenida (n x n)
    
    Raises:
        ValueError: Si alguna de las propiedades no se cumple.
    """
    # Verificar que QR = A
    if not np.allclose(Q @ R, A, atol=1e-10):
        raise ValueError("La propiedad QR = A no se cumple.")
    
    # Verificar que las columnas de Q son ortonormales
    if not np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-10):
        raise ValueError("Las columnas de Q no son ortonormales.")
    
    # Verificar que R es triangular superior con diagonal positiva
    if not np.allclose(R, np.triu(R), atol=1e-10):
        raise ValueError("R no es triangular superior.")
    if not np.all(np.diag(R) > 0):
        raise ValueError("La diagonal de R no es positiva.")

# Verificar ambas descomposiciones
try:
    qrExceptions(A_grande, Q_propia, R_propia)
    print("Descomposición QR propia: Todas las propiedades se cumplen.")
except ValueError as e:
    print(f"Descomposición QR propia: {e}") 

Descomposición QR propia: Todas las propiedades se cumplen.


#### Comparación con Numpy

Ambas son mate´maticamente correctas, solo que numPy utiliza Reflexiones de Househlder, que es un método que no construye los vectores uno a uno limpiándolos sino que aplica transformaciones simétricas, conservando mucho mejor la precisión numérica

Diferencias en los signos;
La descomposición QR no es única en el signo. Si multiplicas una columna de Q por -1 y la fila correspondiente de R por -1 sigue dando A, ambas serán soluciones válidas

### 3.3 Resolución de sistemas lineales con QR

**Ejercicio 3.4:** Implementa una función de sustitución hacia atrás para sistemas triangulares superiores, y úsala para resolver $Ax = b$ vía QR (ver clase 2.2 para el procedimiento).

In [121]:
def sustitucion_atras(R, y):
    """
    Resuelve Rx = y por sustitución hacia atrás, donde R es triangular superior.
    
    Args:
        R: matriz triangular superior (n x n)
        y: vector del lado derecho (array 1D de largo n)
    
    Returns:
        x: solución del sistema
    """
    n = R.shape[0]
    x = np.zeros(n, dtype=float)
 # Empezamos desde la última fila (n-1) hasta la primera (0)
    for i in range(n - 1, -1, -1):
        # El valor de x[i] es el término y[i] menos las x ya calculadas
        suma_conocidos = np.dot(R[i, i+1:], x[i+1:])
        x[i] = (y[i] - suma_conocidos) / R[i, i]
    return x

def resolver_qr(A, b):
    """
    Resuelve Ax = b usando la descomposición QR.
    """
    Q, R = qr_gram_schmidt(A)

    y = Q.T @ b
    x = sustitucion_atras(R, y)
    
    return x

**Ejercicio 3.5:** Prueba `resolver_qr` con el sistema $Ax = b$ donde $A = \begin{bmatrix} 2 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{bmatrix}$ y $b = [5, 16, 41]^\top$.  Compara con `numpy.linalg.solve` y verifica que $Ax = b$.

**Ejercicio 3.6:** Genera una matriz aleatoria invertible de $20 \times 20$ y un vector $b$ aleatorio. Resuelve $Ax = b$ con `resolver_qr` y con `numpy.linalg.solve`.  Compara los resultados y los tiempos de ejecución usando `timeit`.

In [123]:
A = np.array([[2., 1., 1.],
              [4., 3., 3.],
              [8., 7., 9.]])
b = np.array([5., 16., 41.])

# 1. propia
x_propia = resolver_qr(A, b)

# 2. de NumPy
x_numpy = np.linalg.solve(A, b)

print(f"Solución propia: {x_propia}")
print(f"Solución NumPy:  {x_numpy}")
print(f"¿Son iguales?:   {np.allclose(x_propia, x_numpy)}")
print(f"Verificación Ax = b: {np.allclose(A @ x_propia, b)}")

Solución propia: [-0.5  4.5  1.5]
Solución NumPy:  [-0.5  4.5  1.5]
¿Son iguales?:   True
Verificación Ax = b: True


In [125]:
import timeit
A_rand = np.random.rand(20, 20)
b_rand = np.random.rand(20)

# Para medir el tiempo de ejecución de resolver_qr
np.random.seed(42)

# propio
x_qr = resolver_qr(A_rand, b_rand)

# de NumPy
x_np = np.linalg.solve(A_rand, b_rand)

# 3. Comparación de Resultados (Precisión)
distancia = np.linalg.norm(x_qr - x_np)
print(f"Diferencia entre soluciones (Norma): {distancia:.2e}")
print(f"¿Son prácticamente iguales?: {np.allclose(x_qr, x_np)}")

# Ejecutamos 1000 veces cada uno para tener un promedio estable
num_ejecuciones = 1000
tiempo_qr = timeit.timeit(lambda: resolver_qr(A_rand, b_rand), number=num_ejecuciones)
tiempo_np = timeit.timeit(lambda: np.linalg.solve(A_rand, b_rand), number=num_ejecuciones)

print(f"\nTiempo total ({num_ejecuciones} ejecuciones):")
print(f"  Implementación propia (QR): {tiempo_qr:.4f} s")
print(f"  NumPy (linalg.solve):      {tiempo_np:.4f} s")
print(f"Factor de velocidad: NumPy es {tiempo_qr/tiempo_np:.1f}x más rápido.")


"""
Que el algoritmo de numPy sea 90 veces más rápido puede deberse a que está desarrollado en C/Fortran, optimizado para rendimiento
 En contraste, nuestra implementación en Python puro es mucho más lenta debido a la sobrecarga de
"""

Diferencia entre soluciones (Norma): 2.81e-13
¿Son prácticamente iguales?: True

Tiempo total (1000 ejecuciones):
  Implementación propia (QR): 0.6478 s
  NumPy (linalg.solve):      0.0072 s
Factor de velocidad: NumPy es 90.3x más rápido.


---
## 4. Proyecciones usando QR

**Ejercicio 4.1:** Si $B = \hat{Q}\hat{R}$ es la QR reducida de $B$, derivar (en papel o en una celda markdown) que la matriz de proyección sobre $\text{col}(B)$ se simplifica a $P_\pi = \hat{Q}\hat{Q}^\top$.  Partir de la expresión general $P_\pi = B(B^\top B)^{-1}B^\top$ y sustituir $B = \hat{Q}\hat{R}$.

**Ejercicio 4.2:** Implementa una función que calcule la proyección de $x$ sobre $\text{col}(B)$ usando la QR reducida.

![alt text](4f77285f-94ec-4bb7-a0d4-206f7ca51da2.jpg)

In [127]:
def proyeccion_via_qr(B, x):
    """
    Calcula la proyección de x sobre col(B) usando la QR reducida.
    
    Args:
        B: matriz cuyas columnas generan el subespacio (n x m)
        x: vector a proyectar (array 1D de largo n)
    
    Returns:
        proj: la proyección de x
    """
    Q, R = qr_gram_schmidt(B)
    proj = Q @ (Q.T @ x)
    return proj

**Ejercicio 4.3:** Verifica que ambos métodos (ecuación normal y QR) dan el mismo resultado para:

$$B = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}, \quad x = [4, 4, 0]^\top$$

Verificar también usando `coseno` que el residuo es ortogonal a las columnas de $B$ en ambos casos.

In [128]:
B = np.array([[1., 0.],
              [0., 1.],
              [1., 1.]])
x = np.array([4., 4., 0.])

# Proj = B @ inv(B.T @ B) @ B.T @ x
def proyeccion_normal(B, x):
    return B @ np.linalg.inv(B.T @ B) @ B.T @ x

p_normal = proyeccion_normal(B, x)

p_qr = proyeccion_via_qr(B, x)

print("Proyección (Ecuación Normal):", p_normal)
print("Proyección (Vía QR):         ", p_qr)
print("¿Resultados iguales?:", np.allclose(p_normal, p_qr))

# VERIFICACIÓN DEL RESIDUO ORTOGONALIDAD
def verificar_residuo(B, x, p, nombre_metodo):
    residuo = x - p
    print(f"\n--- Análisis de Residuo ({nombre_metodo}) ---")
    for i in range(B.shape[1]):
        col_b = B[:, i]
        # Calcular coseno entre residuo y columna de B
        cos_theta = np.dot(residuo, col_b) / (np.linalg.norm(residuo) * np.linalg.norm(col_b))
        print(f"Coseno con columna {i}: {cos_theta:.2e} (Casi 0)")

verificar_residuo(B, x, p_normal, "Ecuación Normal")
verificar_residuo(B, x, p_qr, "QR")

Proyección (Ecuación Normal): [1.33333333 1.33333333 2.66666667]
Proyección (Vía QR):          [1.33333333 1.33333333 2.66666667]
¿Resultados iguales?: True

--- Análisis de Residuo (Ecuación Normal) ---
Coseno con columna 0: 6.80e-17 (Casi 0)
Coseno con columna 1: 6.80e-17 (Casi 0)

--- Análisis de Residuo (QR) ---
Coseno con columna 0: 6.80e-17 (Casi 0)
Coseno con columna 1: -6.80e-17 (Casi 0)


**Ejercicio 4.4:** Repite la comparación con una matriz aleatoria $B \in \mathbb{R}^{100 \times 5}$ y un vector aleatorio $x \in \mathbb{R}^{100}$.  Compara:
1. Los resultados de ambos métodos (¿coinciden?)
2. Los tiempos de ejecución (usando `timeit`)

¿Cuál de los dos métodos esperarías que sea más rápido? ¿Por qué?

In [ ]:
B_grande = np.random.rand(100, 5)
x_grande = np.random.rand(100)

t_normal = timeit.timeit(lambda: proyeccion_normal(B_grande, x_grande), number=1000)
t_qr = timeit.timeit(lambda: proyeccion_via_qr(B_grande, x_grande), number=1000)

p_normal = proyeccion_normal(B_grande, x_grande)
p_qr = proyeccion_via_qr(B_grande, x_grande)

print(f"¿Los resultados coinciden?: {np.allclose(p_normal, p_qr)}")
print(f"Tiempo Ecuación Normal: {t_normal:.4f} s")
print(f"Tiempo Proyección QR:   {t_qr:.4f} s")


"""
La diferencia no parece tan abismal pero era de esperar que la proyección vía QR es más rápida, 
lo cual es consistente con la idea de que evita la inversión de matrices,
"""

¿Los resultados coinciden?: True
Tiempo Ecuación Normal: 0.0185 s
Tiempo Proyección QR:   0.0555 s


---
## Resumen

En este laboratorio se implementaron:
- Proyecciones ortogonales sobre rectas y subespacios, verificando ortogonalidad del residuo e idempotencia de la matriz de proyección.
- El algoritmo de Gram-Schmidt para construir bases ortonormales.
- La descomposición QR como subproducto de Gram-Schmidt, y su uso para resolver sistemas lineales.
- Una formulación alternativa de las proyecciones usando QR ($\pi = QQ^\top x$), que evita la inversión de $B^\top B$.